Key Deliverables 
- EntityExtractor class with regex patterns for numeric entities ✅
- Amenity detection using Week 1 taxonomy ✅
- Labeled dataset: 200-300 remarks with entity spans 
- Evaluation script calculating precision/recall/F1 
- Target: 85%+ F1 score on test set 
- Error analysis identifying top failure patterns 

Objective: Build a program that reads a real estate listing description/remarks and pull out important information into a structured format

In [1]:
import re 
from collections import Counter
import json
import pandas as pd
import sys
sys.path.append("../scripts")

In [2]:
from w2_text_cleaning import TextCleaner

========== REMARKS PROFILE ==========

Total listings: 1000
Null rate: 0.00%
Average length: 1290.09 characters

Price mentions: 57
Measurement mentions: 261
Room dimensions: 7

HTML tags: 0
Unicode usages: 394
Smart quotes: 276
Whitespace issues: 482
APN Counts: 3
Phone Number Counts: 0

Known abbreviations
ft       251
sq       237
condo    139
hoa      134
adu      122
bbq      104
rv       70
ev       58
hvac     57
ac       45

Unknown abbreviations
altos    3
sony     3
ages     3
wings    3
pulls    3
hemet    3
harte    3
rents    3
sj       3
tools    3
finds    3
items    3
lofts    3
gpm      2
draws    2
verde    2
lies     2
rises    2
tells    2
baja     2

Top 20 words
home         2383
living       1964
room         1233
offers       1073
space        1068
kitchen      953
private      849
bedroom      832
s            824
dining       815
2            796
features     761
you          750
area         690
located      668
spacious     667
bedrooms     666
new          

In [3]:
with open("../data/processed/taxonomy.json", "r") as f:
    taxonomy = json.load(f)
print("Total terms:", len(taxonomy["terms"]))
print(taxonomy["categories"])
print()
term_counts = Counter(term["category"] for term in taxonomy["terms"])

print("Term Counts per Category")
for category in taxonomy["categories"]:
    count = term_counts.get(category, 0)
    print(f"{category}: {count} terms")

Total terms: 200
['kitchen', 'flooring', 'outdoor', 'parking', 'housing_layout', 'housing_features', 'location', 'community_amenities']

Term Counts per Category
kitchen: 22 terms
flooring: 13 terms
outdoor: 31 terms
parking: 20 terms
housing_layout: 43 terms
housing_features: 41 terms
location: 28 terms
community_amenities: 2 terms


In [4]:
print(taxonomy['terms'])

[{'id': 1, 'term': 'primary suite', 'category': 'housing_layout', 'frequency': 336}, {'id': 2, 'term': 'natural light', 'category': 'housing_features', 'frequency': 294}, {'id': 3, 'term': 'living room', 'category': 'housing_layout', 'frequency': 283}, {'id': 4, 'term': 'living space', 'category': 'outdoor', 'frequency': 263}, {'id': 5, 'term': 'floor plan', 'category': 'flooring', 'frequency': 261}, {'id': 6, 'term': 'stainless steel', 'category': 'kitchen', 'frequency': 193}, {'id': 7, 'term': 'everyday living', 'category': 'parking', 'frequency': 186}, {'id': 8, 'term': 'stainless steel appliances', 'category': 'kitchen', 'frequency': 172}, {'id': 9, 'term': 'shopping dining', 'category': 'location', 'frequency': 171}, {'id': 10, 'term': 'located near', 'category': 'location', 'frequency': 159}, {'id': 11, 'term': 'conveniently located', 'category': 'location', 'frequency': 158}, {'id': 12, 'term': 'home office', 'category': 'housing_layout', 'frequency': 143}, {'id': 13, 'term': 'p

In [274]:
class EntityExtractor: 
    def __init__(self):
        with open("../data/processed/taxonomy.json", "r") as f:
            taxonomy = json.load(f)
        self.taxonomy = taxonomy['terms']
    def extract_bedrooms(self, text): 
        patterns = [
        # Numeric values
        r'(\d+)\s*[- ]?(?:\w+\s+){0,5}?(?:bed|beds|bedroom|bedrooms|br|bd|bdrm|bdrms|suite|suites)\b',
        # Number words
        r'(one|two|three|four|five|six|seven|eight|nine|ten)\s+(?:\w+\s+){0,3}?(?:bed|beds|bedroom|bedrooms|br|bd|bdrm|bdrms)\b'
        ]
        number_words = {
        "one": 1,
        "two": 2,
        "three": 3,
        "four": 4,
        "five": 5,
        "six": 6,
        "seven": 7,
        "eight": 8,
        "nine": 9,
        "ten": 10
        }
        # Exact bedroom counts
        for pattern in patterns: 
            match = re.search(pattern, text, re.I) 
            if match: 
                value = match.group(1).lower()
                if value.isdigit():
                    return int(value)
                return number_words[value]
            
        # Ordinal bedroom references
        ordinal_map = {
        "primary": 1,
        "first": 1,
        "second": 2,
        "third": 3,
        "fourth": 4,
        "fifth": 5,
        "sixth": 6,
        "seventh": 7,
        "eighth": 8,
        "ninth": 9,
        "tenth": 10
        }
        matches = re.findall(
            r'\b(primary|first|second|third|fourth|fifth|sixth|seventh|eighth|ninth|tenth)\s+(?:bedroom|suite)\b',
            text,
            re.I
        )
        if matches:
            return max(ordinal_map[word.lower()] for word in matches)
        
        # Plural bedroom mention
        if re.search(r'\b(beds|bedrooms)\b', text, re.I):
            return ">1"
        
        # Singular bedroom mention
        if re.search(r'\b(bed|bedroom)\b', text, re.I):
            return 1
        return None 
  
    def extract_price(self, text): 
        if not isinstance(text, str):
            return None
        match = re.search(
            r'\$?\s*([\d,]{5,})',
            text
        )
        if match:
            return int(match.group(1).replace(",", ""))
        return None
    def extract_bathrooms(self, text):
        patterns =  [
            r'(\d+\.?\d*)\s*(?:bath|baths|bathroom|bathrooms|ba)\b'
        ]
        for pattern in patterns: 
            match = re.search(pattern, text, re.I) 
            if match:
                value = float(match.group(1))
                if value.is_integer():
                    return int(value)
                return value
        return None 
    def extract_sqft(self, text): 
        if not isinstance(text, str):
            return None
        patterns = [
            r'([\d,]+)\s*(?:square\s*feet)\b',
            r'([\d,]+)\s*(?:sq\.?\s*ft\.?)\b',
            r'([\d,]+)\s*sqft\b',
            r'([\d,]+)\s*sf\b'
        ]
        for pattern in patterns:
            match = re.search(pattern, text, re.I)
            if match:
                return int(
                    match.group(1).replace(",", "")
                )
        return None
    def extract_amenities(self, text):
        amenities = []
        text = text.lower()
        for item in self.taxonomy:
            term = item["term"].lower()
            if term in text:
                amenities.append({
                    "term": item["term"],
                    "category": item["category"]
                })
        return amenities
    def extract_all(self, text): 
        return { 
        'bedrooms': self.extract_bedrooms(text), 
        'bathrooms': self.extract_bathrooms(text), 
        'price': self.extract_price(text), 
        'sqft': self.extract_sqft(text), 
        'amenities': self.extract_amenities(text) 
        } 


Test cases

In [275]:
extractor = EntityExtractor()

In [202]:
remark_ex = '''
This condominium is set up as a great roommate split. Two primary bedroom suites, each on its own floor...
'''
print(extractor.extract_bedrooms(remark_ex))

2


In [204]:
remark_ex2 = '''
Welcome to La Playa Court. Built in 2008, this sophisticated building offers upscale elegance and resort-style amenities just minutes from the beach, local restaurants, and shops. This elegant, rare corner unit features a spacious open floor plan designed for both entertaining and privacy. Upon entering the unit, you come upon the dining area with large window and shutters for privacy. The gourmet chef's kitchen is cleverly set off the dining area and equipped with high-end appliances, granite countertops, travertine stone backsplash and flooring. There is even a window above the sink and a breakfast bar. The open living area is centered by a modern and elegant gas fireplace and access to a private balcony. The primary suite includes a custom walk-in closet and an ensuite spa bath with dual sinks, a soaking tub, and a separate shower. The second bedroom also features a large wall-to-wall closet, with a second full bathroom and linen closet located nearby,
'''
print(extractor.extract_bedrooms(remark_ex2))

2


In [205]:
remark_ex3 = '''
Welcome to this stunning, move-in-ready residence showcasing modern curb appeal and thoughtfully updated interiors throughout. The striking black-and-white exterior creates a sophisticated first impression, complemented by a spacious front yard, custom walkway accents, and an oversized two-car garage. Inside, you'll find an open and airy floor plan featuring vaulted ceilings, contemporary gray wood-look flooring, fresh paint, and abundant natural light. The spacious living area is highlighted by a stylish painted brick fireplace and large sliding glass doors that seamlessly connect indoor and outdoor living spaces. The remodeled kitchen and bathrooms feature elegant finishes, including modern fixtures, upgraded cabinetry, quartz-style countertops, and a sleek vessel sink in the guest bath. Generously sized bedrooms and updated living spaces provide comfort and functionality for today's lifestyle. The private backyard offers endless possibilities for entertaining, relaxation, or future customization.
'''
print(extractor.extract_bedrooms(remark_ex3))

>1


In [206]:
remark_ex4 = ''' 
*AMAZING LOG-STYLE RETREAT IN EAGLE MOUNTAIN ESTATES!** This beautifully maintained single-story home offers level entry, 3 spacious en suite bedrooms (3 master suites), 3 baths, and approximately 2,200 square feet. of comfortable mountain living. The open great room features soaring cathedral ceilings, abundant natural light, and a stunning custom stone fireplace. Situated on a large, partially fenced **10,000 square feet. lot**, the property offers plenty of outdoor space, including a covered front porch and a large refinished rear deck with inviting views. Additional features include an **oversized 2-car attached garage**, **Electric vehicle parking**, and a **tankless water heater**. Recent upgrades include a new Aspen spa, A/C system, whole-house humidifier, garage door, and solar system (**solar negotiable**). An unfinished area beneath the home provides excellent additional storage potential. Ideally located near Big Bear Lake, ski resorts, hiking trails, shopping, and dining. Perfect as a full-time residence, vacation getaway, or short-term rental opportunity. **Furnishings negotiable.** A wonderful mountain escape ready to create lasting memories with family and friends.
'''
print(extractor.extract_bedrooms(remark_ex4))

3


In [207]:
remark_ex5 = '''Featuring 3 spacious bedrooms'''
print(extractor.extract_bedrooms(remark_ex5))

3


In [208]:
remark_ex6 = '''The spacious bedroom and well-kept living room'''
print(extractor.extract_bedrooms(remark_ex6))

1


In [7]:
df_cleaned = pd.read_csv("../data/processed/cleaned_listing_sample.csv")
df_cleaned.head()

,L_ListingID,L_Address,L_City,beds,baths,price,remarks,cleaned_remarks
0,1169503734,133 Crystal Street,Taft,3.0,2.0,235000,This unique property offers two homes on one l...,This unique property offers two homes on one l...
1,1154220129,15664 Kadota,Victorville,4.0,4.0,459000,Beautiful Two-Story Home in Victorville – Spac...,Beautiful Two-Story Home in Victorville - Spac...
2,1159921951,949 S Manhattan Place 203,Los Angeles,3.0,2.0,689000,Presenting this exquisite second-floor corner ...,Presenting this exquisite second-floor corner ...
3,1170333192,1872 Seascape Boulevard,Aptos,3.0,3.0,1195000,Thoughtfully renovated coastal retreat tucked ...,Thoughtfully renovated coastal retreat tucked ...
4,1152463169,2085 Westhampton Drive,Arroyo Grande,3.0,2.0,1050000,Welcome to 2085 Westhampton Drive in Arroyo Gr...,Welcome to 2085 Westhampton Drive in Arroyo Gr...


In [8]:
df_cleaned[['remarks']].head(1)

,remarks
0,This unique property offers two homes on one l...


Checking extracting amenities

In [9]:
remark = """
Beautiful 4 bedroom home with granite countertops,
luxury vinyl plank flooring, pool, and RV parking.
"""

In [10]:
print(extractor.extract_amenities(remark))

[{'term': 'granite countertops', 'category': 'kitchen'}, {'term': 'plank flooring', 'category': 'flooring'}, {'term': 'vinyl plank flooring', 'category': 'flooring'}, {'term': 'rv parking', 'category': 'parking'}]


Extracting all

In [11]:
df_cleaned.columns

Index(['L_ListingID', 'L_Address', 'L_City', 'beds', 'baths', 'price',
       'remarks', 'cleaned_remarks'],
      dtype='str')

In [12]:
sample_remark = df_cleaned['cleaned_remarks'].iloc[0]

In [13]:
sample_remark

'This unique property offers two homes on one lot, creating an exceptional opportunity for both owner-occupants and investors alike. The front home features 2 bedrooms and 1 bathroom, while the rear unit offers 1 bedroom and 1 bathroom. Ideal for extended family, rental income, or a live-in-one-rent-the-other setup. The owner currently lives in one unit, which makes it easier to occupy or rent it out!'

In [14]:
print(extractor.extract_all(sample_remark))

{'bedrooms': 2, 'bathrooms': 1, 'price': None, 'sqft': None, 'amenities': []}


With this sample, only the first mentions of the bedrooms and bathrooms were extracted. Looking at the remark, they expanded on what the rear unit consists of. 

In [15]:
example_remark = '''
There are 2.5 beds and 3 bathrooms.
'''

In [16]:
print(extractor.extract_all(sample_remark))

{'bedrooms': 2, 'bathrooms': 1, 'price': None, 'sqft': None, 'amenities': []}


#### Labeled dataset: 200-300 remarks with entity spans 

In [220]:
labeled_sample = df_cleaned.sample(n=400, random_state=422)

In [221]:
#labeled_sample.to_csv('../data/labeled_sample.csv', index=False)
print(len(labeled_sample))

400


In [222]:
labeled_sample_cleaned = labeled_sample.drop(columns=['L_Address', 'remarks', 'L_City', 'L_ListingID'])

In [223]:
labeled_sample_cleaned.head()

,beds,baths,price,cleaned_remarks
606,2.0,1.0,399977,"This inviting 2-bedroom, 1-bath residence sits on a generous 1/3-acre lot in the desirable north downtown area of Yucca Valley, offering a blend of comfort, functionality, and seclusion. The property is framed by attractive landscaping and scenic mountain views, with privacy fencing enclosing both the front and rear yards. A gated outdoor parking area adds convenience and security, while covered patios at the front and back create perfect spots to unwind and enjoy the surroundings. Inside, the home features a bright and open living space highlighted by vaulted, beamed ceilings, tile flooring throughout, and updated windows that bring in plenty of natural light. The kitchen is well-equipped and includes a casual dining nook with sliding glass doors leading to the backyard, along with a separate dining area ideal for entertaining or flexible use. The spacious bathroom includes a tub and shower combination. One bedroom offers built-in storage and comfortable proportions, while the second bedroom is notably large, filled with natural light, and includes a generous closet. An additional bonus room provides extra space suitable for a home office, creative studio, or guest accommodations. This home is ready for immediate enjoyment. With its well-designed layout, recent improvements, and inviting outdoor spaces, this property presents a fantastic opportunity in a sought-after Yucca Valley location. Come tour this property today!"
255,2.0,3.0,389000,"Multi-level townhouse looking for a little love. This condominium is set up as a great roommate split. Two primary bedroom suites, each on its own floor. Main floor has full sized kitchen, living/dining room with fireplace and deck and a half bathroom. Two car garage. Private fenced backyard area. Close to downtown Redlands and an easy commute to Loma Linda. Redlands school district. Homeowners Association includes pool and spa. Unit #D."
258,4.0,4.0,2625000,"Experience refined Central Coast living in this stunning 4,146 square feet. Spanish Colonial estate, perfectly positioned on a private 3 acre parcel within the exclusive gates of Santa Ysabel Ranch. Enjoy sweeping 180° views of vineyards and mountains from nearly every room. This 4-bedroom, 3.5-bath home welcomes you through a grand rotunda entry with soaring ceilings, custom lighting, and travertine flooring throughout. The formal living and dining rooms feature tall Anderson windows framing breathtaking vistas and a cozy gas fireplace. The gourmet kitchen is equipped with GE Monogram stainless appliances, including a six-burner range, double ovens, wine fridge, and granite countertops with maple cabinetry. The adjoining family room and casual dining area create an inviting space for entertaining with seamless indoor-outdoor flow. The spacious primary suite offers privacy, dual walk-in closets, and a luxurious bath with soaking tub and large shower. Additional bedrooms and a flexible office space capture beautiful hillside and sunset views. Outdoor living shines with expansive flagstone patios, multiple sitting areas, and lush landscaping surrounded by heritage oaks - perfect for dining, relaxing, or entertaining under the stars. Additional features include an epoxy-finished 3-car garage with custom cabinetry, Culligan water system, American district telegraph security, and custom lighting throughout. Santa Ysabel Ranch offers 24/7 gated security, walking trails, tennis courts, and a serene lake setting - an unparalleled blend of luxury, privacy, and natural beauty."
301,3.0,4.0,3325000,"Wine country estate set across 52 acres in the heart of the Paso Robles AVA, commanding unobstructed views over 28.5 planted acres of Cabernet Sauvignon and expansive sunset horizons. The 3,869 square feet. custom residence is thoughtfully positioned to capture the surrounding vineyard vistas from nearly every room. A dramatic entry introduces Travertine flooring, exposed beams, and walls of windo

sqft (ground truth)

In [224]:
labeled_sample_cleaned['sqft'] = None

In [225]:
sqft_rows = labeled_sample_cleaned[
    labeled_sample_cleaned["cleaned_remarks"].str.contains(
        r'\d[\d,]*(?:[±+]|\+/-)?\s*(?:square\s+feet|sq\s*ft|sqft|\bsf\b)',
        case=False,
        na=False,
        regex=True
    )
]
len(sqft_rows)

139

In [226]:
def sqft_context(text, window=35):
    if pd.isna(text):
        return None

    match = re.search(
        r'\d[\d,]*(?:[±+]|\+/-)?\s*(?:square\s+feet|sq\s*ft|sqft|\bsf\b)',
        text,
        flags=re.IGNORECASE
    )

    if not match:
        return None

    start = max(0, match.start() - window)
    end = min(len(text), match.end() + window)

    return text[start:end]

In [227]:
labeled_sample_cleaned["sqft_context"] = labeled_sample_cleaned["cleaned_remarks"].apply(sqft_context)

In [228]:
labeled_sample_cleaned[labeled_sample_cleaned["sqft_context"].notna()][["sqft_context"]]

,sqft_context
258,"tral Coast living in this stunning 4,146 square feet. Spanish Colonial estate, perfectl"
301,"and expansive sunset horizons. The 3,869 square feet. custom residence is thoughtfully"
944,"nal versatility. Inside, more than 1,500 square feet of light filled living space showc"
288,"ring 5 bedrooms, 4.5 bathrooms and 3,606 square feet with a bright open layout and SPAC"
229,"nt single-family home spans nearly 3,300 square feet. and offers a perfect blend of com"
...,...
922,"N END UNIT, THIS Condominium BOAST 2167 square feet WITH MANY COMMUNITY AMNETIES, INCL"
275,"5-bedroom, 3.5-bath home offering 3,012 square feet. of living space on a lush and pri"
492,"full bathrooms, and approximately 1,318 square feet of single-level living space, with"
848,e ranging from 2 to 4 bedrooms and 1435 square feet to 2339 square feet. Stop on by an


In [229]:
def sqft_phrase(text):
    if pd.isna(text):
        return None

    match = re.search(
        r'\d[\d,]*(?:[±+]|\+/-)?\s*(?:square\s+feet|sq\s*ft|sqft|\bsf\b)',
        text,
        flags=re.IGNORECASE
    )

    return match.group(0) if match else None

In [230]:
labeled_sample_cleaned["sqft_phrase"] = labeled_sample_cleaned["cleaned_remarks"].apply(sqft_phrase)

In [231]:
labeled_sample_cleaned['sqft_phrase'].notna().sum()

np.int64(139)

In [232]:
labeled_sample_cleaned["sqft_phrase"] = labeled_sample_cleaned["cleaned_remarks"].apply(sqft_phrase)
labeled_sample_cleaned["sqft_context"] = labeled_sample_cleaned["cleaned_remarks"].apply(sqft_context)
labeled_sample_cleaned["sqft"] = None

In [233]:
df_cleaned['cleaned_remarks'].loc[258]

'Experience refined Central Coast living in this stunning 4,146 square feet. Spanish Colonial estate, perfectly positioned on a private 3 acre parcel within the exclusive gates of Santa Ysabel Ranch. Enjoy sweeping 180° views of vineyards and mountains from nearly every room. This 4-bedroom, 3.5-bath home welcomes you through a grand rotunda entry with soaring ceilings, custom lighting, and travertine flooring throughout. The formal living and dining rooms feature tall Anderson windows framing breathtaking vistas and a cozy gas fireplace. The gourmet kitchen is equipped with GE Monogram stainless appliances, including a six-burner range, double ovens, wine fridge, and granite countertops with maple cabinetry. The adjoining family room and casual dining area create an inviting space for entertaining with seamless indoor-outdoor flow. The spacious primary suite offers privacy, dual walk-in closets, and a luxurious bath with soaking tub and large shower. Additional bedrooms and a flexible

In [234]:
labeled_sample_cleaned.head(10)

,beds,baths,price,cleaned_remarks,sqft,sqft_context,sqft_phrase
606,2.0,1.0,399977,"This inviting 2-bedroom, 1-bath residence sits on a generous 1/3-acre lot in the desirable north downtown area of Yucca Valley, offering a blend of comfort, functionality, and seclusion. The property is framed by attractive landscaping and scenic mountain views, with privacy fencing enclosing both the front and rear yards. A gated outdoor parking area adds convenience and security, while covered patios at the front and back create perfect spots to unwind and enjoy the surroundings. Inside, the home features a bright and open living space highlighted by vaulted, beamed ceilings, tile flooring throughout, and updated windows that bring in plenty of natural light. The kitchen is well-equipped and includes a casual dining nook with sliding glass doors leading to the backyard, along with a separate dining area ideal for entertaining or flexible use. The spacious bathroom includes a tub and shower combination. One bedroom offers built-in storage and comfortable proportions, while the second bedroom is notably large, filled with natural light, and includes a generous closet. An additional bonus room provides extra space suitable for a home office, creative studio, or guest accommodations. This home is ready for immediate enjoyment. With its well-designed layout, recent improvements, and inviting outdoor spaces, this property presents a fantastic opportunity in a sought-after Yucca Valley location. Come tour this property today!",None,NaN,NaN
255,2.0,3.0,389000,"Multi-level townhouse looking for a little love. This condominium is set up as a great roommate split. Two primary bedroom suites, each on its own floor. Main floor has full sized kitchen, living/dining room with fireplace and deck and a half bathroom. Two car garage. Private fenced backyard area. Close to downtown Redlands and an easy commute to Loma Linda. Redlands school district. Homeowners Association includes pool and spa. Unit #D.",None,NaN,NaN
258,4.0,4.0,2625000,"Experience refined Central Coast living in this stunning 4,146 square feet. Spanish Colonial estate, perfectly positioned on a private 3 acre parcel within the exclusive gates of Santa Ysabel Ranch. Enjoy sweeping 180° views of vineyards and mountains from nearly every room. This 4-bedroom, 3.5-bath home welcomes you through a grand rotunda entry with soaring ceilings, custom lighting, and travertine flooring throughout. The formal living and dining rooms feature tall Anderson windows framing breathtaking vistas and a cozy gas fireplace. The gourmet kitchen is equipped with GE Monogram stainless appliances, including a six-burner range, double ovens, wine fridge, and granite countertops with maple cabinetry. The adjoining family room and casual dining area create an inviting space for entertaining with seamless indoor-outdoor flow. The spacious primary suite offers privacy, dual walk-in closets, and a luxurious bath with soaking tub and large shower. Additional bedrooms and a flexible office space capture beautiful hillside and sunset views. Outdoor living shines with expansive flagstone patios, multiple sitting areas, and lush landscaping surrounded by heritage oaks - perfect for dining, relaxing, or entertaining under the stars. Additional features include an epoxy-finished 3-car garage with custom cabinetry, Culligan water system, American district telegraph security, and custom lighting throughout. Santa Ysabel Ranch offers 24/7 gated security, walking trails, tennis courts, and a serene lake setting - an unparalleled blend of luxury, privacy, and natural beauty.",None,"tral Coast living in this stunning 4,146 square feet. Spanish Colonial estate, perfectl","4,146 square feet"
301,3.0,4.0,3325000,"Wine country estate set across 52 acres in the heart of the Paso Robles AVA, commanding unobstructed views over 28.5 planted acres of Cabernet Sauvignon and expansive sunset horizons. The 3,869 square feet. custom residence is th

In [235]:
def label_sqft(phrase):
    if pd.isna(phrase):
        return None

    match = re.search(r'(\d[\d,]*)', phrase)

    if match:
        return int(match.group(1).replace(",", ""))

    return None

In [236]:
labeled_sample_cleaned["sqft"] = labeled_sample_cleaned["sqft_phrase"].apply(label_sqft)

In [237]:
labeled_sample_cleaned['sqft'].notna().sum()

np.int64(139)

In [238]:
labeled_sample_cleaned.columns

Index(['beds', 'baths', 'price', 'cleaned_remarks', 'sqft', 'sqft_context',
       'sqft_phrase'],
      dtype='str')

In [239]:
labeled_sample_cleaned['sqft_context'].notna().sum()

np.int64(139)

In [240]:
labeled_sample_cleaned['sqft_phrase'].notna().sum()

np.int64(139)

In [241]:
labeled_cleaned = labeled_sample_cleaned.drop(columns=['sqft_phrase', 'sqft_context'])

In [242]:
labeled_cleaned["sqft"] = labeled_cleaned["sqft"].fillna("None")

In [243]:
labeled_cleaned['sqft'].isna().sum()

np.int64(0)

Amendities (ground truth)

In [244]:
# Using taxonomy as a helper/resource
def suggest_amenities(text):
    suggestions = []

    for term in taxonomy_terms:
        if term.lower() in text.lower():
            suggestions.append(term)

    return suggestions

taxonomy_terms = sorted(
    [term["term"] for term in taxonomy["terms"]],
    key=len,
    reverse=True
)

In [245]:
labeled_cleaned["candidate_amenities"] = (
    labeled_cleaned["cleaned_remarks"]
    .apply(suggest_amenities)
)

In [246]:
labeled_cleaned['candidate_amenities']

606                                                                                                                              [flooring throughout, sliding glass doors, mountain views, second bedroom, natural light, tile flooring, covered patio, outdoor space, sliding glass, living space, home office, glass doors]
255                                                                                                                                                                                                                                                                                 [primary bedroom, dining room, car garage]
258                                                               [spacious primary suite, primary suite offers, granite countertops, flooring throughout, additional bedrooms, spacious primary, gourmet kitchen, primary suite, tennis courts, gas fireplace, suite offers, family room, dining room, car garage, bath home]
301                                        

Go through the suggested amenities

In [247]:
#labeled_cleaned.to_csv("../data/labeled_cleaned.csv")

Keep in mind: The amentities are meant to represent the features that the property offers, not surrounding words or phrases that happen to contain those features. I will also avoid any marketing phrases. An amenity will represent a tangible property feature, space, service, or location advantage that the property provides. So, if there are any phrases that are not supportive of this, they will be removed (to keep ground truth)

In [248]:
all_amenities = [
    amenity
    for row in labeled_cleaned["candidate_amenities"]
    for amenity in row
]

amenity_counts = Counter(all_amenities)

amenity_counts.most_common()

[('living space', 138),
 ('primary suite', 130),
 ('natural light', 110),
 ('floor plan', 104),
 ('car garage', 98),
 ('living room', 96),
 ('everyday living', 71),
 ('conveniently located', 64),
 ('full bath', 63),
 ('stainless steel', 62),
 ('located near', 61),
 ('easy access', 59),
 ('stainless steel appliances', 56),
 ('home office', 51),
 ('family room', 50),
 ('recessed lighting', 49),
 ('living spaces', 48),
 ('quartz countertops', 45),
 ('full bathroom', 45),
 ('laundry room', 42),
 ('conveniently located near', 41),
 ('dining room', 39),
 ('granite countertops', 38),
 ('abundant natural light', 37),
 ('ideally located', 36),
 ('outdoor space', 35),
 ('suite offers', 35),
 ('bath home', 35),
 ('fully remodeled', 35),
 ('spacious primary', 34),
 ('gated community', 34),
 ('spacious living', 33),
 ('main level', 33),
 ('open floor', 32),
 ('primary bedroom', 31),
 ('primary suite offers', 31),
 ('direct access', 30),
 ('covered patio', 29),
 ('spacious primary suite', 29),
 ('fi

In [249]:
len(amenity_counts)

156

In [250]:
removed_amenities= [
 'everyday living', 'conveniently located', 'located near', 
 'easy access', 'conveniently located near', 'ideally located', 
 'spacious primary', 'flooring throughout', 'fully remodeled', 
 'primary suite offers', 'suite offers', 'direct access', 'flows seamlessly', 
 'convenient access', 'everyday convenience', 'located within', 
 'prime location', 'brand new', 'ample space', 'home located', 
 'beautifully remodeled', 'ideally located near', 'comfortable living space', 
 'enjoy access', 'endless possibilities', 'seamless flow', 'effortless entertaining', 
 'primary suite serves', 'suite serves', 'kitchen showcases', 'everyday comfort', 
 'every detail', 'primary suite features', 'suite features','suite includes', 
 'floor plan features', 'peaceful retreat', 'kitchen featuring', 'level features', 
 'spacious home', 'bright open', 'patio perfect', 'kitchen equipped',
 'features spacious','bedrooms including', 'seamless living', 'located minutes',
 'feet living space', 'space home'
]

In [251]:
labeled_cleaned["candidate_amenities"] = labeled_cleaned["candidate_amenities"].apply(
    lambda amenities: [
        x for x in amenities
        if x not in removed_amenities
    ]
)

In [252]:
all_amenities = [
    amenity
    for row in labeled_cleaned["candidate_amenities"]
    for amenity in row
]

amenity_counts = Counter(all_amenities)

amenity_counts.most_common()

[('living space', 138),
 ('primary suite', 130),
 ('natural light', 110),
 ('floor plan', 104),
 ('car garage', 98),
 ('living room', 96),
 ('full bath', 63),
 ('stainless steel', 62),
 ('stainless steel appliances', 56),
 ('home office', 51),
 ('family room', 50),
 ('recessed lighting', 49),
 ('living spaces', 48),
 ('quartz countertops', 45),
 ('full bathroom', 45),
 ('laundry room', 42),
 ('dining room', 39),
 ('granite countertops', 38),
 ('abundant natural light', 37),
 ('outdoor space', 35),
 ('bath home', 35),
 ('gated community', 34),
 ('spacious living', 33),
 ('main level', 33),
 ('open floor', 32),
 ('primary bedroom', 31),
 ('covered patio', 29),
 ('spacious primary suite', 29),
 ('fitness center', 29),
 ('kitchen features', 29),
 ('updated kitchen', 28),
 ('open floor plan', 28),
 ('parking space', 27),
 ('golf course', 27),
 ('vaulted ceilings', 26),
 ('mountain views', 25),
 ('wood flooring', 25),
 ('private balcony', 25),
 ('hardwood floors', 25),
 ('center island', 25)

In [253]:
len(amenity_counts)

108

In [254]:
labeled_cleaned = labeled_cleaned.rename(columns={'candidate_amenities': 'amenities'})

In [255]:
labeled_cleaned.head()

,beds,baths,price,cleaned_remarks,sqft,amenities
606,2.0,1.0,399977,"This inviting 2-bedroom, 1-bath residence sits on a generous 1/3-acre lot in the desirable north downtown area of Yucca Valley, offering a blend of comfort, functionality, and seclusion. The property is framed by attractive landscaping and scenic mountain views, with privacy fencing enclosing both the front and rear yards. A gated outdoor parking area adds convenience and security, while covered patios at the front and back create perfect spots to unwind and enjoy the surroundings. Inside, the home features a bright and open living space highlighted by vaulted, beamed ceilings, tile flooring throughout, and updated windows that bring in plenty of natural light. The kitchen is well-equipped and includes a casual dining nook with sliding glass doors leading to the backyard, along with a separate dining area ideal for entertaining or flexible use. The spacious bathroom includes a tub and shower combination. One bedroom offers built-in storage and comfortable proportions, while the second bedroom is notably large, filled with natural light, and includes a generous closet. An additional bonus room provides extra space suitable for a home office, creative studio, or guest accommodations. This home is ready for immediate enjoyment. With its well-designed layout, recent improvements, and inviting outdoor spaces, this property presents a fantastic opportunity in a sought-after Yucca Valley location. Come tour this property today!",None,"[sliding glass doors, mountain views, second bedroom, natural light, tile flooring, covered patio, outdoor space, sliding glass, living space, home office, glass doors]"
255,2.0,3.0,389000,"Multi-level townhouse looking for a little love. This condominium is set up as a great roommate split. Two primary bedroom suites, each on its own floor. Main floor has full sized kitchen, living/dining room with fireplace and deck and a half bathroom. Two car garage. Private fenced backyard area. Close to downtown Redlands and an easy commute to Loma Linda. Redlands school district. Homeowners Association includes pool and spa. Unit #D.",None,"[primary bedroom, dining room, car garage]"
258,4.0,4.0,2625000,"Experience refined Central Coast living in this stunning 4,146 square feet. Spanish Colonial estate, perfectly positioned on a private 3 acre parcel within the exclusive gates of Santa Ysabel Ranch. Enjoy sweeping 180° views of vineyards and mountains from nearly every room. This 4-bedroom, 3.5-bath home welcomes you through a grand rotunda entry with soaring ceilings, custom lighting, and travertine flooring throughout. The formal living and dining rooms feature tall Anderson windows framing breathtaking vistas and a cozy gas fireplace. The gourmet kitchen is equipped with GE Monogram stainless appliances, including a six-burner range, double ovens, wine fridge, and granite countertops with maple cabinetry. The adjoining family room and casual dining area create an inviting space for entertaining with seamless indoor-outdoor flow. The spacious primary suite offers privacy, dual walk-in closets, and a luxurious bath with soaking tub and large shower. Additional bedrooms and a flexible office space capture beautiful hillside and sunset views. Outdoor living shines with expansive flagstone patios, multiple sitting areas, and lush landscaping surrounded by heritage oaks - perfect for dining, relaxing, or entertaining under the stars. Additional features include an epoxy-finished 3-car garage with custom cabinetry, Culligan water system, American district telegraph security, and custom lighting throughout. Santa Ysabel Ranch offers 24/7 gated security, walking trails, tennis courts, and a serene lake setting - an unparalleled blend of luxury, privacy, and natural beauty.",4146.0,"[spacious primary suite, granite countertops, additional bedrooms, gourmet kitchen, primary suite, tennis courts, gas fireplace, family room, dining room, car garage, bath 

Using the extractor class

In [276]:
labeled_extract = labeled_cleaned.copy()

In [277]:
labeled_extract.shape

(400, 6)

Extracting bedrooms

In [278]:
labeled_extract['extracted_bd'] = labeled_extract['cleaned_remarks'].apply(
    extractor.extract_bedrooms
)

In [279]:
labeled_extract['extracted_bd'].isna().sum()

np.int64(53)

In [280]:
missing_bd = labeled_extract[
    labeled_extract["extracted_bd"].isna()
].copy()

In [281]:
missing_bd.shape

(53, 7)

In [282]:
bed_pattern = (
    r"\b("
    r"bed|beds|bedroom|bedrooms|"
    r"br|bd|bdrm|bdrms"
    r")\b"
)

mentioned_bed = missing_bd[
    missing_bd["cleaned_remarks"].str.contains(
        bed_pattern,
        case=False,
        regex=True,
        na=False
    )
]

C:\Users\mayab\AppData\Local\Temp\ipykernel_4572\2472845788.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  missing_bd["cleaned_remarks"].str.contains(


In [283]:
mentioned_bed.shape

(0, 7)

In [284]:
mentioned_bed.columns

Index(['beds', 'baths', 'price', 'cleaned_remarks', 'sqft', 'amenities',
       'extracted_bd'],
      dtype='str')

In [285]:
for i, row in mentioned_bed.iterrows():
    print(f"Row {i}")
    print(f"Ground truth beds: {row['beds']}")
    print(f"Extracted beds: {row['extracted_bd']}")
    print(row["cleaned_remarks"])
    print("-" * 100)

Now that there are no more bedroom mentions, we can assume 53 remarks did not mention anything about bedrooms, so we will remove those rows from our analysis. 

In [286]:
labeled_extract = labeled_extract[~labeled_extract["extracted_bd"].isna()]

In [287]:
labeled_extract.shape

(347, 7)

Extracting bathrooms

In [297]:
labeled_extract['extracted_br'] = labeled_extract['cleaned_remarks'].apply(
    extractor.extract_bathrooms
)

In [298]:
labeled_extract['extracted_br'].isna().sum()

np.int64(240)

Extracting price

In [295]:
labeled_extract['extracted_price'] = labeled_extract['cleaned_remarks'].apply(
    extractor.extract_price
)

In [296]:
labeled_extract['extracted_price'].isna().sum()

np.int64(200)

Extracting sqft

In [293]:
labeled_extract['extracted_sqft'] = labeled_extract['cleaned_remarks'].apply(
    extractor.extract_sqft
)

In [294]:
labeled_extract['extracted_sqft'].isna().sum()

np.int64(219)

Extracting amenities

In [291]:
labeled_extract['extracted_amenities'] = labeled_extract['cleaned_remarks'].apply(
    extractor.extract_amenities
)

In [292]:
labeled_extract['extracted_amenities'].isna().sum()

np.int64(0)